# Ingeniería de características

Este notebook construye el conjunto de variables para modelar fraude transaccional a partir de `card_transactions_clean.parquet`.

El flujo queda organizado para que cada bloque tenga una responsabilidad clara:

1. cargar y validar la base limpia;
2. crear variables transaccionales, temporales y de canal;
3. calcular historiales acumulados sin usar la transacción actual;
4. agregar ventanas mensuales anteriores al mes de evaluación de cada transacción;
5. crear ratios, flags de novedad, velocidad y rareza;
6. validar nulos/infinitos y exportar los datasets de modelado.


## 1. Configuración

Se importan librerías, se fijan rutas del proyecto y se declaran parámetros reutilizables. El modo muestra permite probar el flujo con uno o varios grupos del parquet sin cargar toda la base.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Encuentra la ra?z del proyecto desde el cwd del notebook o del repositorio."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No se pudo ubicar la ra?z del proyecto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = DATA_PROCESSED_DIR / "card_transactions_clean.parquet"
OUTPUT_TRANSACTION_FEATURES_PATH = DATA_PROCESSED_DIR / "transactions_modeling.parquet"
OUTPUT_USER_CARD_SNAPSHOT_PATH = DATA_PROCESSED_DIR / "user_card_snapshot_features.parquet"

USE_SAMPLE = False
SAMPLE_ROW_GROUPS = [0]
WINDOW_MONTHS = [3, 6, 9, 12]
RARE_FREQUENCY_THRESHOLD = 0.0001
SAVE_OUTPUTS = True

INPUT_PATH


## 2. Carga y validación inicial

La validación se hace antes de transformar para asegurar que la base de entrada contiene las columnas mínimas esperadas por el pipeline.


In [ ]:
def read_transactions(path: Path, use_sample: bool = False, row_groups: list[int] | None = None) -> pd.DataFrame:
    """Carga la base completa o una muestra por grupos de filas del parquet."""
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo de entrada: {path}")

    if not use_sample:
        return pd.read_parquet(path)

    import pyarrow as pa
    import pyarrow.parquet as pq

    parquet_file = pq.ParquetFile(path)
    selected_groups = row_groups or [0]
    tables = [parquet_file.read_row_group(group) for group in selected_groups]
    return pa.concat_tables(tables).to_pandas()


In [ ]:
df = read_transactions(
    INPUT_PATH,
    use_sample=USE_SAMPLE,
    row_groups=SAMPLE_ROW_GROUPS,
)

df.shape


In [ ]:
required_columns = [
    "user",
    "card",
    "year",
    "month",
    "day",
    "time",
    "amount",
    "amount_abs",
    "amount_is_negative",
    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "is_fraud",
    "merchant_state_was_missing",
    "zip_was_missing",
    "user_card_id",
    "hour",
    "datetime",
    "day_of_week",
    "is_weekend",
]

missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Columnas faltantes en la base limpia: {missing_columns}")

print("Base validada sin columnas faltantes.")


## 3. Variables base

Este bloque normaliza fechas, crea columnas limpias para categorías con posibles nulos y construye variables simples de monto, canal y calendario.


In [ ]:
df = df.copy()
df["_row_order"] = np.arange(len(df), dtype="int64")

df["datetime"] = pd.to_datetime(df["datetime"])
df["year_month"] = df["datetime"].dt.to_period("M").dt.to_timestamp()
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["quarter"] = df["datetime"].dt.quarter
df["day_of_month"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

evaluation_month = df["year_month"].max()
evaluation_month


In [ ]:
clean_columns = {
    "merchant_name": "merchant_name_clean",
    "merchant_city": "merchant_city_clean",
    "merchant_state": "merchant_state_clean",
    "zip": "zip_clean",
    "mcc": "mcc_clean",
    "use_chip": "use_chip_clean",
}

for raw_col, clean_col in clean_columns.items():
    df[clean_col] = df[raw_col].fillna("UNKNOWN").astype(str)


In [ ]:
df["amount_log"] = np.log1p(df["amount_abs"])
df["amount_is_zero_flag"] = (df["amount_abs"] == 0).astype(int)
df["amount_below_1_flag"] = (df["amount_abs"] < 1).astype(int)
df["amount_below_5_flag"] = (df["amount_abs"] < 5).astype(int)
df["amount_is_round_10_flag"] = np.isclose(df["amount_abs"] % 10, 0, atol=1e-6).astype(int)
df["amount_is_round_100_flag"] = np.isclose(df["amount_abs"] % 100, 0, atol=1e-6).astype(int)

amount_quantiles = df["amount_abs"].quantile([0.90, 0.95, 0.99])
df["amount_above_global_p90_flag"] = (df["amount_abs"] > amount_quantiles.loc[0.90]).astype(int)
df["amount_above_global_p95_flag"] = (df["amount_abs"] > amount_quantiles.loc[0.95]).astype(int)
df["amount_above_global_p99_flag"] = (df["amount_abs"] > amount_quantiles.loc[0.99]).astype(int)


In [ ]:
use_chip_lower = df["use_chip_clean"].str.lower()

df["is_online_transaction"] = use_chip_lower.str.contains("online", na=False).astype(int)
df["is_chip_transaction"] = use_chip_lower.str.contains("chip", na=False).astype(int)
df["is_swipe_transaction"] = use_chip_lower.str.contains("swipe", na=False).astype(int)

df["is_night"] = df["hour"].between(0, 5).astype(int)
df["is_morning"] = df["hour"].between(6, 11).astype(int)
df["is_afternoon"] = df["hour"].between(12, 17).astype(int)
df["is_evening"] = df["hour"].between(18, 23).astype(int)
df["is_business_hours"] = df["hour"].between(8, 18).astype(int)
df["is_late_night"] = df["hour"].between(0, 3).astype(int)


## 4. Historial acumulado por tarjeta

Todas las variables históricas usan información estrictamente anterior a la transacción actual. Por eso se ordena por `user_card_id`, `datetime` y el orden original de la fila antes de usar acumulados.


In [ ]:
def safe_divide(numerator, denominator, fill_value: float = 0.0):
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan).fillna(fill_value)


df = df.sort_values(["user_card_id", "datetime", "_row_order"]).reset_index(drop=True)

df["uc_tx_count_hist"] = df.groupby("user_card_id").cumcount()
df["uc_amount_sum_hist"] = df.groupby("user_card_id")["amount_abs"].cumsum() - df["amount_abs"]
df["uc_amount_mean_hist"] = safe_divide(df["uc_amount_sum_hist"], df["uc_tx_count_hist"])
df["uc_amount_max_hist"] = (
    df.groupby("user_card_id")["amount_abs"]
    .cummax()
    .groupby(df["user_card_id"])
    .shift(1)
    .fillna(0)
)

df["uc_prev_datetime"] = df.groupby("user_card_id")["datetime"].shift(1)
df["uc_days_since_prev_tx"] = (
    (df["datetime"] - df["uc_prev_datetime"])
    .dt.total_seconds()
    .div(86_400)
    .fillna(-1)
)


In [ ]:
df["uc_no_history_flag"] = (df["uc_tx_count_hist"] == 0).astype(int)
df["first_transaction_flag"] = (df["uc_days_since_prev_tx"] == -1).astype(int)
df["long_inactivity_30d_flag"] = (df["uc_days_since_prev_tx"] > 30).astype(int)
df["long_inactivity_90d_flag"] = (df["uc_days_since_prev_tx"] > 90).astype(int)

df["amount_to_hist_mean_ratio"] = safe_divide(df["amount_abs"], df["uc_amount_mean_hist"])
df["amount_above_hist_max_flag"] = (
    (df["uc_amount_max_hist"] > 0) & (df["amount_abs"] > df["uc_amount_max_hist"])
).astype(int)
df["amount_gt_3x_hist_mean_flag"] = (
    (df["uc_amount_mean_hist"] > 0) & (df["amount_abs"] > 3 * df["uc_amount_mean_hist"])
).astype(int)


## 5. Ventanas mensuales sin fuga de información

Las ventanas móviles se calculan por meses calendario completos anteriores al mes de cada transacción. Para evitar sesgos cuando una tarjeta no tiene transacciones en algún mes, se completa la grilla mensual antes de aplicar `shift(1)` y `rolling`.


In [ ]:
def build_monthly_user_card_features(transactions: pd.DataFrame, windows: list[int]) -> pd.DataFrame:
    monthly_source = transactions[[
        "user_card_id",
        "year_month",
        "amount_abs",
        "is_online_transaction",
        "amount_is_negative",
        "merchant_name_clean",
        "mcc_clean",
    ]].copy()
    monthly_source["amount_abs_sq"] = monthly_source["amount_abs"] ** 2

    monthly = (
        monthly_source
        .groupby(["user_card_id", "year_month"], observed=True)
        .agg(
            uc_month_tx_count=("amount_abs", "size"),
            uc_month_amount_sum=("amount_abs", "sum"),
            uc_month_amount_sq_sum=("amount_abs_sq", "sum"),
            uc_month_amount_min=("amount_abs", "min"),
            uc_month_amount_max=("amount_abs", "max"),
            uc_month_online_tx_count=("is_online_transaction", "sum"),
            uc_month_negative_tx_count=("amount_is_negative", "sum"),
            uc_month_unique_merchants=("merchant_name_clean", "nunique"),
            uc_month_unique_mcc=("mcc_clean", "nunique"),
        )
        .reset_index()
    )

    user_cards = pd.Index(transactions["user_card_id"].drop_duplicates(), name="user_card_id")
    months = pd.date_range(
        transactions["year_month"].min(),
        transactions["year_month"].max(),
        freq="MS",
        name="year_month",
    )
    full_index = pd.MultiIndex.from_product([user_cards, months])

    monthly = (
        monthly
        .set_index(["user_card_id", "year_month"])
        .reindex(full_index)
        .reset_index()
        .sort_values(["user_card_id", "year_month"])
    )

    zero_fill_cols = [
        "uc_month_tx_count",
        "uc_month_amount_sum",
        "uc_month_amount_sq_sum",
        "uc_month_online_tx_count",
        "uc_month_negative_tx_count",
        "uc_month_unique_merchants",
        "uc_month_unique_mcc",
    ]
    monthly[zero_fill_cols] = monthly[zero_fill_cols].fillna(0)

    grouped = monthly.groupby("user_card_id", sort=False)
    for window in windows:
        tx_col = f"uc_tx_count_{window}m"
        sum_col = f"uc_amount_sum_{window}m"
        sq_sum_col = f"uc_amount_sq_sum_{window}m"
        mean_col = f"uc_amount_mean_{window}m"
        std_col = f"uc_amount_std_{window}m"

        monthly[tx_col] = grouped["uc_month_tx_count"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[sum_col] = grouped["uc_month_amount_sum"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[sq_sum_col] = grouped["uc_month_amount_sq_sum"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[f"uc_amount_min_{window}m"] = grouped["uc_month_amount_min"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).min()
        )
        monthly[f"uc_amount_max_{window}m"] = grouped["uc_month_amount_max"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).max()
        )
        monthly[f"uc_online_tx_count_{window}m"] = grouped["uc_month_online_tx_count"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[f"uc_negative_tx_count_{window}m"] = grouped["uc_month_negative_tx_count"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[f"uc_unique_merchants_{window}m"] = grouped["uc_month_unique_merchants"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )
        monthly[f"uc_unique_mcc_{window}m"] = grouped["uc_month_unique_mcc"].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).sum()
        )

        monthly[mean_col] = safe_divide(monthly[sum_col], monthly[tx_col])
        variance = (
            monthly[sq_sum_col] - (monthly[sum_col] ** 2 / monthly[tx_col].replace(0, np.nan))
        ) / (monthly[tx_col] - 1).replace(0, np.nan)
        monthly[std_col] = np.sqrt(variance.clip(lower=0)).fillna(0)

        monthly[f"uc_avg_ticket_{window}m"] = safe_divide(monthly[sum_col], monthly[tx_col])
        monthly[f"uc_amount_cv_{window}m"] = safe_divide(monthly[std_col], monthly[mean_col])
        monthly[f"uc_amount_max_mean_ratio_{window}m"] = safe_divide(
            monthly[f"uc_amount_max_{window}m"], monthly[mean_col]
        )
        monthly[f"uc_amount_min_mean_ratio_{window}m"] = safe_divide(
            monthly[f"uc_amount_min_{window}m"], monthly[mean_col]
        )
        monthly[f"uc_online_tx_rate_{window}m"] = safe_divide(
            monthly[f"uc_online_tx_count_{window}m"], monthly[tx_col]
        )
        monthly[f"uc_negative_tx_rate_{window}m"] = safe_divide(
            monthly[f"uc_negative_tx_count_{window}m"], monthly[tx_col]
        )

    return monthly.replace([np.inf, -np.inf], np.nan).fillna(0)


In [ ]:
monthly_user_card = build_monthly_user_card_features(df, WINDOW_MONTHS)
monthly_user_card.shape


In [ ]:
extended_window_columns = []
for window in WINDOW_MONTHS:
    extended_window_columns.extend([
        f"uc_tx_count_{window}m",
        f"uc_amount_sum_{window}m",
        f"uc_amount_mean_{window}m",
        f"uc_amount_min_{window}m",
        f"uc_amount_max_{window}m",
        f"uc_amount_std_{window}m",
        f"uc_avg_ticket_{window}m",
        f"uc_amount_cv_{window}m",
        f"uc_amount_max_mean_ratio_{window}m",
        f"uc_amount_min_mean_ratio_{window}m",
        f"uc_online_tx_count_{window}m",
        f"uc_online_tx_rate_{window}m",
        f"uc_negative_tx_count_{window}m",
        f"uc_negative_tx_rate_{window}m",
        f"uc_unique_merchants_{window}m",
        f"uc_unique_mcc_{window}m",
    ])

monthly_user_card["uc_tx_count_ratio_3m_6m"] = safe_divide(
    monthly_user_card["uc_tx_count_3m"], monthly_user_card["uc_tx_count_6m"]
)
monthly_user_card["uc_tx_count_ratio_3m_12m"] = safe_divide(
    monthly_user_card["uc_tx_count_3m"], monthly_user_card["uc_tx_count_12m"]
)
monthly_user_card["uc_tx_count_ratio_6m_12m"] = safe_divide(
    monthly_user_card["uc_tx_count_6m"], monthly_user_card["uc_tx_count_12m"]
)
monthly_user_card["uc_amount_sum_ratio_3m_6m"] = safe_divide(
    monthly_user_card["uc_amount_sum_3m"], monthly_user_card["uc_amount_sum_6m"]
)
monthly_user_card["uc_amount_sum_ratio_3m_12m"] = safe_divide(
    monthly_user_card["uc_amount_sum_3m"], monthly_user_card["uc_amount_sum_12m"]
)
monthly_user_card["uc_amount_sum_ratio_6m_12m"] = safe_divide(
    monthly_user_card["uc_amount_sum_6m"], monthly_user_card["uc_amount_sum_12m"]
)
monthly_user_card["uc_amount_mean_delta_3m_12m"] = (
    monthly_user_card["uc_amount_mean_3m"] - monthly_user_card["uc_amount_mean_12m"]
)
monthly_user_card["uc_amount_mean_ratio_3m_12m"] = safe_divide(
    monthly_user_card["uc_amount_mean_3m"], monthly_user_card["uc_amount_mean_12m"]
)

extended_change_columns = [
    "uc_tx_count_ratio_3m_6m",
    "uc_tx_count_ratio_3m_12m",
    "uc_tx_count_ratio_6m_12m",
    "uc_amount_sum_ratio_3m_6m",
    "uc_amount_sum_ratio_3m_12m",
    "uc_amount_sum_ratio_6m_12m",
    "uc_amount_mean_delta_3m_12m",
    "uc_amount_mean_ratio_3m_12m",
]

monthly_user_card["uc_no_tx_3m_flag"] = (monthly_user_card["uc_tx_count_3m"] == 0).astype(int)
monthly_user_card["uc_no_tx_6m_flag"] = (monthly_user_card["uc_tx_count_6m"] == 0).astype(int)
monthly_user_card["uc_activity_spike_3m_vs_12m_flag"] = (
    monthly_user_card["uc_tx_count_ratio_3m_12m"] > 0.60
).astype(int)
monthly_user_card["uc_amount_spike_3m_vs_12m_flag"] = (
    monthly_user_card["uc_amount_sum_ratio_3m_12m"] > 0.60
).astype(int)
monthly_user_card["uc_mean_amount_increase_3m_vs_12m_flag"] = (
    monthly_user_card["uc_amount_mean_ratio_3m_12m"] > 1.50
).astype(int)

extended_change_columns.extend([
    "uc_no_tx_3m_flag",
    "uc_no_tx_6m_flag",
    "uc_activity_spike_3m_vs_12m_flag",
    "uc_amount_spike_3m_vs_12m_flag",
    "uc_mean_amount_increase_3m_vs_12m_flag",
])


In [ ]:
monthly_user_card_features = monthly_user_card[
    ["user_card_id", "year_month"] + extended_window_columns + extended_change_columns
].copy()

duplicate_feature_cols = [
    col for col in monthly_user_card_features.columns
    if col in df.columns and col not in ["user_card_id", "year_month"]
]
if duplicate_feature_cols:
    df = df.drop(columns=duplicate_feature_cols)

df = df.merge(
    monthly_user_card_features,
    on=["user_card_id", "year_month"],
    how="left",
    validate="many_to_one",
)

df[extended_window_columns + extended_change_columns] = (
    df[extended_window_columns + extended_change_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
df = df.copy()


## 6. Comparaciones de monto contra históricos

Se comparan los montos de cada transacción contra el histórico acumulado y contra ventanas de 3, 6, 9 y 12 meses.


In [ ]:
amount_feature_data = {}

for window in WINDOW_MONTHS:
    mean_col = f"uc_amount_mean_{window}m"
    max_col = f"uc_amount_max_{window}m"
    std_col = f"uc_amount_std_{window}m"

    amount_feature_data[f"amount_to_{window}m_mean_ratio"] = safe_divide(df["amount_abs"], df[mean_col])
    amount_feature_data[f"amount_to_{window}m_max_ratio"] = safe_divide(df["amount_abs"], df[max_col])
    amount_feature_data[f"amount_minus_{window}m_mean"] = df["amount_abs"] - df[mean_col]
    amount_feature_data[f"amount_zscore_{window}m"] = safe_divide(df["amount_abs"] - df[mean_col], df[std_col])
    amount_feature_data[f"amount_above_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > df[mean_col])
    ).astype(int)
    amount_feature_data[f"amount_above_2x_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > 2 * df[mean_col])
    ).astype(int)
    amount_feature_data[f"amount_above_3x_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > 3 * df[mean_col])
    ).astype(int)
    amount_feature_data[f"amount_above_{window}m_max_flag"] = (
        (df[max_col] > 0) & (df["amount_abs"] > df[max_col])
    ).astype(int)

amount_features = pd.DataFrame(amount_feature_data, index=df.index)
df = pd.concat([df, amount_features], axis=1)
df["amount_gt_3x_12m_mean_flag"] = df["amount_above_3x_12m_mean_flag"]

amount_window_comparison_cols = [
    col for col in df.columns
    if (
        col.startswith("amount_to_")
        or col.startswith("amount_minus_")
        or col.startswith("amount_zscore_")
        or col.startswith("amount_above_")
    )
]

df[amount_window_comparison_cols] = (
    df[amount_window_comparison_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
df = df.copy()


## 7. Historial de canal y novedad comercial

Estos atributos capturan si una tarjeta está usando por primera vez un canal, comercio, MCC o ubicación. Los conteos por par se calculan antes de la transacción actual mediante `cumcount()`.


In [ ]:
df = df.sort_values(["user_card_id", "datetime", "_row_order"]).reset_index(drop=True)

channel_columns = [
    ("is_online_transaction", "online"),
    ("is_chip_transaction", "chip"),
    ("is_swipe_transaction", "swipe"),
]

channel_hist_cols = []
channel_feature_data = {}

for source_col, channel_name in channel_columns:
    count_col = f"uc_{channel_name}_tx_count_hist"
    rate_col = f"uc_{channel_name}_tx_rate_hist"
    first_col = f"{channel_name}_first_time_for_card_flag"

    count_values = df.groupby("user_card_id")[source_col].cumsum() - df[source_col]
    channel_feature_data[count_col] = count_values
    channel_feature_data[rate_col] = safe_divide(count_values, df["uc_tx_count_hist"])
    channel_feature_data[first_col] = ((df[source_col] == 1) & (count_values == 0)).astype(int)
    channel_hist_cols.extend([count_col, rate_col, first_col])

df = pd.concat([df, pd.DataFrame(channel_feature_data, index=df.index)], axis=1)


In [ ]:
novelty_specs = [
    ("merchant_name_clean", "new_merchant_for_card_flag", "uc_merchant_tx_count_hist", "uc_unique_merchants_hist"),
    ("mcc_clean", "new_mcc_for_card_flag", "uc_mcc_tx_count_hist", "uc_unique_mcc_hist"),
    ("merchant_city_clean", "new_city_for_card_flag", "uc_city_tx_count_hist", "uc_unique_cities_hist"),
    ("merchant_state_clean", "new_state_for_card_flag", "uc_state_tx_count_hist", "uc_unique_states_hist"),
    ("zip_clean", "new_zip_for_card_flag", "uc_zip_tx_count_hist", "uc_unique_zips_hist"),
]

novelty_cols = []
novelty_count_cols = []
unique_history_cols = []
novelty_feature_data = {}

for source_col, new_col, count_col, unique_col in novelty_specs:
    count_values = df.groupby(["user_card_id", source_col]).cumcount()
    new_values = (count_values == 0).astype(int)

    novelty_feature_data[count_col] = count_values
    novelty_feature_data[new_col] = new_values
    novelty_feature_data[unique_col] = new_values.groupby(df["user_card_id"]).cumsum() - new_values

    novelty_cols.append(new_col)
    novelty_count_cols.append(count_col)
    unique_history_cols.append(unique_col)

df = pd.concat([df, pd.DataFrame(novelty_feature_data, index=df.index)], axis=1)
df = df.copy()


## 8. Historial por usuario

Además del historial por tarjeta, se agregan señales acumuladas a nivel de usuario para capturar comportamiento multi-tarjeta sin usar el identificador como predictor directo.


In [ ]:
cards_per_user = df.groupby("user")["card"].nunique().rename("n_cards_user")
df = df.merge(cards_per_user, on="user", how="left", validate="many_to_one")

df = df.sort_values(["user", "datetime", "_row_order"]).reset_index(drop=True)

user_tx_count_hist = df.groupby("user").cumcount()
user_amount_sum_hist = df.groupby("user")["amount_abs"].cumsum() - df["amount_abs"]
user_amount_mean_hist = safe_divide(user_amount_sum_hist, user_tx_count_hist)
user_amount_max_hist = (
    df.groupby("user")["amount_abs"]
    .cummax()
    .groupby(df["user"])
    .shift(1)
    .fillna(0)
)
user_prev_datetime = df.groupby("user")["datetime"].shift(1)
user_days_since_prev_tx = (
    (df["datetime"] - user_prev_datetime)
    .dt.total_seconds()
    .div(86_400)
    .fillna(-1)
)

user_feature_data = {
    "user_tx_count_hist": user_tx_count_hist,
    "user_amount_sum_hist": user_amount_sum_hist,
    "user_amount_mean_hist": user_amount_mean_hist,
    "user_amount_max_hist": user_amount_max_hist,
    "user_days_since_prev_tx": user_days_since_prev_tx,
    "amount_to_user_hist_mean_ratio": safe_divide(df["amount_abs"], user_amount_mean_hist),
    "amount_to_user_hist_max_ratio": safe_divide(df["amount_abs"], user_amount_max_hist),
    "amount_above_user_hist_max_flag": (
        (user_amount_max_hist > 0) & (df["amount_abs"] > user_amount_max_hist)
    ).astype(int),
}

df = pd.concat([df, pd.DataFrame(user_feature_data, index=df.index)], axis=1)

user_hist_cols = list(user_feature_data.keys())
df = df.copy()


## 9. Velocidad intradía

Estas variables miden cuánta actividad previa tiene la misma tarjeta durante el mismo día de la transacción.


In [ ]:
df = df.sort_values(["user_card_id", "datetime", "_row_order"]).reset_index(drop=True)
df["transaction_date"] = df["datetime"].dt.date

same_day_tx_count_prev = df.groupby(["user_card_id", "transaction_date"]).cumcount()
same_day_amount_sum_prev = (
    df.groupby(["user_card_id", "transaction_date"])["amount_abs"].cumsum() - df["amount_abs"]
)

velocity_feature_data = {
    "uc_same_day_tx_count_prev": same_day_tx_count_prev,
    "uc_same_day_amount_sum_prev": same_day_amount_sum_prev,
    "uc_same_day_amount_mean_prev": safe_divide(same_day_amount_sum_prev, same_day_tx_count_prev),
    "multiple_tx_same_day_flag": (same_day_tx_count_prev >= 1).astype(int),
    "high_velocity_same_day_flag": (same_day_tx_count_prev >= 5).astype(int),
    "very_high_velocity_same_day_flag": (same_day_tx_count_prev >= 10).astype(int),
}

df = pd.concat([df, pd.DataFrame(velocity_feature_data, index=df.index)], axis=1)
velocity_cols = list(velocity_feature_data.keys())


## 10. Frecuencias globales y rareza

Para variables categóricas de alta cardinalidad se agregan conteos/frecuencias globales. Después se crean flags de rareza con un umbral explícito y fácil de ajustar.


In [ ]:
def build_global_frequency_features(
    data: pd.DataFrame,
    specs: list[tuple[str, str, str]],
) -> pd.DataFrame:
    frequency_feature_data = {}
    for source_col, count_col, frequency_col in specs:
        counts = data[source_col].value_counts(dropna=False)
        count_values = data[source_col].map(counts).fillna(0).astype("int64")
        frequency_feature_data[count_col] = count_values
        frequency_feature_data[frequency_col] = count_values / len(data)
    return pd.DataFrame(frequency_feature_data, index=data.index)


frequency_specs = [
    ("merchant_name_clean", "merchant_tx_count_global", "merchant_frequency_global"),
    ("mcc_clean", "mcc_tx_count_global", "mcc_frequency_global"),
    ("merchant_state_clean", "merchant_state_tx_count_global", "merchant_state_frequency_global"),
    ("merchant_city_clean", "merchant_city_tx_count_global", "merchant_city_frequency_global"),
    ("zip_clean", "zip_tx_count_global", "zip_frequency_global"),
    ("use_chip_clean", "use_chip_tx_count_global", "use_chip_frequency_global"),
]

frequency_features = build_global_frequency_features(df, frequency_specs)
df = pd.concat([df, frequency_features], axis=1)
frequency_cols = list(frequency_features.columns)


In [ ]:
rare_specs = [
    ("merchant_frequency_global", "rare_merchant_flag"),
    ("mcc_frequency_global", "rare_mcc_flag"),
    ("merchant_city_frequency_global", "rare_city_flag"),
    ("merchant_state_frequency_global", "rare_state_flag"),
    ("zip_frequency_global", "rare_zip_flag"),
]

rare_feature_data = {
    flag_col: (df[frequency_col] < RARE_FREQUENCY_THRESHOLD).astype(int)
    for frequency_col, flag_col in rare_specs
}

df = pd.concat([df, pd.DataFrame(rare_feature_data, index=df.index)], axis=1)
rare_cols = list(rare_feature_data.keys())
df = df.copy()


## 11. Catálogo de variables y validación final

Se centraliza la lista de variables numéricas, se eliminan duplicados conservando el orden y se valida que no queden nulos ni infinitos en las columnas de modelado.


In [ ]:
base_numeric_features = [
    "amount_abs",
    "amount_log",
    "amount_is_negative",
    "merchant_state_was_missing",
    "zip_was_missing",
    "hour",
    "day_of_week",
    "is_weekend",
    "quarter",
    "day_of_month",
    "n_cards_user",
    "is_online_transaction",
    "is_chip_transaction",
    "is_swipe_transaction",
    "is_night",
    "is_morning",
    "is_afternoon",
    "is_evening",
    "is_business_hours",
    "is_late_night",
    "amount_above_global_p90_flag",
    "amount_above_global_p95_flag",
    "amount_above_global_p99_flag",
    "amount_is_zero_flag",
    "amount_below_1_flag",
    "amount_below_5_flag",
    "amount_is_round_10_flag",
    "amount_is_round_100_flag",
    "uc_tx_count_hist",
    "uc_amount_sum_hist",
    "uc_amount_mean_hist",
    "uc_amount_max_hist",
    "uc_days_since_prev_tx",
    "uc_no_history_flag",
    "first_transaction_flag",
    "long_inactivity_30d_flag",
    "long_inactivity_90d_flag",
    "amount_to_hist_mean_ratio",
    "amount_above_hist_max_flag",
    "amount_gt_3x_hist_mean_flag",
    "amount_gt_3x_12m_mean_flag",
]

numeric_features = (
    base_numeric_features
    + extended_window_columns
    + extended_change_columns
    + amount_window_comparison_cols
    + channel_hist_cols
    + novelty_cols
    + novelty_count_cols
    + unique_history_cols
    + user_hist_cols
    + velocity_cols
    + frequency_cols
    + rare_cols
)

numeric_features = list(dict.fromkeys(numeric_features))
missing_features = [col for col in numeric_features if col not in df.columns]
if missing_features:
    raise ValueError(f"Variables numéricas no creadas: {missing_features}")

len(numeric_features)


In [ ]:
df[numeric_features] = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

feature_nulls = df[numeric_features].isna().sum().sort_values(ascending=False)
non_finite_counts = pd.Series({
    col: np.isinf(df[col].to_numpy(dtype="float64", copy=False)).sum()
    for col in numeric_features
}).sort_values(ascending=False)

assert feature_nulls.sum() == 0
assert non_finite_counts.sum() == 0

pd.DataFrame({
    "n_rows": [len(df)],
    "n_numeric_features": [len(numeric_features)],
    "fraud_rate": [df["is_fraud"].mean()],
    "evaluation_month": [evaluation_month],
})


In [ ]:
feature_catalog = pd.DataFrame({
    "feature": numeric_features,
    "dtype": [str(df[col].dtype) for col in numeric_features],
    "missing_values": [df[col].isna().sum() for col in numeric_features],
})

feature_catalog.head(20)


## 12. Exportación

El dataset transaccional conserva identificadores, fecha, target y variables numéricas listas para modelado. El snapshot por tarjeta toma la última observación disponible de cada `user_card_id`.


In [ ]:
identifier_columns = [
    "user",
    "card",
    "user_card_id",
    "datetime",
    "year_month",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "use_chip",
]
target_columns = ["is_fraud"]

modeling_columns = list(dict.fromkeys(identifier_columns + target_columns + numeric_features))
df_modeling = df[modeling_columns].sort_values("datetime").reset_index(drop=True)

user_card_snapshot = (
    df.sort_values(["user_card_id", "datetime", "_row_order"])
    .groupby("user_card_id", as_index=False)
    .tail(1)
    .loc[:, ["user", "card", "user_card_id", "datetime", "year_month"] + numeric_features]
    .rename(columns={"datetime": "snapshot_datetime", "year_month": "snapshot_month"})
    .reset_index(drop=True)
)

df_modeling.shape, user_card_snapshot.shape


In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_TRANSACTION_FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_modeling.to_parquet(OUTPUT_TRANSACTION_FEATURES_PATH, index=False)
    user_card_snapshot.to_parquet(OUTPUT_USER_CARD_SNAPSHOT_PATH, index=False)

    print(f"Dataset transaccional guardado en: {OUTPUT_TRANSACTION_FEATURES_PATH}")
    print(f"Snapshot por tarjeta guardado en: {OUTPUT_USER_CARD_SNAPSHOT_PATH}")
else:
    print("SAVE_OUTPUTS=False: no se escribieron archivos.")
